In [1]:
!pip install transformers datasets accelerate -q


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from datasets import Dataset

from sklearn.metrics import accuracy_score, f1_score

In [3]:
train_df = pd.read_csv("../data/processed/train_processed.csv")
valid_df = pd.read_csv("../data/processed/valid_processed.csv")

print(train_df.shape)
print(valid_df.shape)

(9539, 3)
(2388, 3)


In [ ]:
train_df = train_df.sample(2000, random_state=42)
valid_df = valid_df.sample(500, random_state=42)

In [5]:
model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [6]:


def tokenize_function(example):

    return tokenizer(
        example["clean_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [7]:
train_dataset = Dataset.from_pandas(
    train_df[["clean_text", "label"]]
)

valid_dataset = Dataset.from_pandas(
    valid_df[["clean_text", "label"]]
)

In [8]:
train_dataset = train_dataset.map(tokenize_function)

valid_dataset = valid_dataset.map(tokenize_function)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [9]:
train_dataset = train_dataset.rename_column("label", "labels")
valid_dataset = valid_dataset.rename_column("label", "labels")

In [10]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

valid_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [11]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    f1 = f1_score(
        labels,
        predictions,
        average="weighted"
    )

    return {
        "accuracy": accuracy,
        "f1": f1
    }

In [12]:
training_args = TrainingArguments(

    output_dir="../outputs/finbert_results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_dir="../outputs/logs",

    load_best_model_at_end=True,

    metric_for_best_model="f1"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [13]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    compute_metrics=compute_metrics
)

In [14]:
trainer.train()

c:\Desktop copy\financial-news-sentiment-analysis\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.628872,0.754000,0.718266
2,0.625760,0.668087,0.778000,0.773045
3,0.625760,0.748196,0.778000,0.776972


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Desktop copy\financial-news-sentiment-analysis\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Desktop copy\financial-news-sentiment-analysis\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=750, training_loss=0.5242887369791667, metrics={'train_runtime': 3309.1205, 'train_samples_per_second': 1.813, 'train_steps_per_second': 0.227, 'total_flos': 394670126592000.0, 'train_loss': 0.5242887369791667, 'epoch': 3.0})

In [15]:
for i in range(20):
    print(train_df["text"].iloc[i])
    print("LABEL:", train_df["label"].iloc[i])
    print("-"*50)

WATCH - ‘We did not initiate this trade war and this is not something we want’: Chinese President Xi Jinping says h… https://t.co/tgMUQllm6M
LABEL: 2
--------------------------------------------------
Ex-Pimco CEO Douglas Hodge Gets Nine Months in College Admission Scam
LABEL: 2
--------------------------------------------------
$SVXY - Visualizing The Predictive Value Of Contango. Get more updates here: https://t.co/RdAshxEAcB #trading #business #investing
LABEL: 2
--------------------------------------------------
Ilkka-Yhtymä Oyj: Johdon liiketoimet #Stock #MarketScreener https://t.co/Je2UYbTKuP https://t.co/CLCJGNePOA
LABEL: 2
--------------------------------------------------
What Is The TED Spread Telling Investors About The Credit Market?
LABEL: 2
--------------------------------------------------
CORRECTED-Ex-British Airways executive indicted over alleged JFK Airport bribery scheme
LABEL: 2
--------------------------------------------------
Avid Tech +11.5% after upside EPS fo

In [16]:
for label in [0, 1, 2]:

    print(f"\n===== LABEL {label} =====\n")

    samples = train_df[train_df["label"] == label]["text"].head(10)

    for text in samples:
        print(text)
        print()


===== LABEL 0 =====

Wholesale inventories in the U.S. slip 0.2% in December

$ING - ING May Be Through The Worst Point Of The Cycle. Read more: https://t.co/wTGUTspx6E #stockmarket #economy #investing

Hopes Dim for Broad Deal on Global Carbon Market at UN Talks

FX strategists aren’t expecting a repeat of the loonie’s outperformance in 2020, casting doubt on its run as one of… https://t.co/peQZYOcMhB

Ealing’s downturn hits high-priced houses https://t.co/WLEnqLdOf1

In case you need another reason to delete your Facebook: https://t.co/YEEHiptvAV

Stocks have given back most of their gains, and the Nasdaq has turned negative https://t.co/sXyTbUf0jW

Stocks give up gains. Experts weigh in on the state of the market (via @TradingNation) https://t.co/oC0O8aXDeR

New lawsuit seeks to pin blame for 737 MAX on Boeing's board

DMCI continues slide as government cancels extension of water concession deals - BusinessWorld Online


===== LABEL 1 =====

Avid Tech +11.5% after upside EPS foreca

In [17]:
model.save_pretrained(
    "../models/bert/finbert_model",
    safe_serialization=False
)

tokenizer.save_pretrained(
    "../models/bert/finbert_tokenizer"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('../models/bert/finbert_tokenizer\\tokenizer_config.json',
 '../models/bert/finbert_tokenizer\\tokenizer.json')

In [23]:
import torch

label_map = {
    0: "Bearish",
    1: "Bullish",
    2: "Neutral"
}

def predict_finbert(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():

        outputs = model(**inputs)

        probs = torch.nn.functional.softmax(
            outputs.logits,
            dim=-1
        )

        prediction = torch.argmax(probs).item()

        confidence = torch.max(probs).item()

    return {
        "sentiment": label_map[prediction],
        "confidence": round(confidence, 4)
    }

In [24]:
predict_finbert(
    "Tesla stock surged after record quarterly earnings"
)

{'sentiment': 'Bullish', 'confidence': 0.6771}

In [25]:
predict_finbert(
    "The company suffered massive financial losses"
)

{'sentiment': 'Bearish', 'confidence': 0.838}

In [26]:
predict_finbert(
    "The board announced its annual meeting"
)

{'sentiment': 'Neutral', 'confidence': 0.992}

In [27]:
import os

save_path = "../models/bert/finbert_model"

os.makedirs(save_path, exist_ok=True)

torch.save(
    model.state_dict(),
    os.path.join(save_path, "pytorch_model.bin")
)

model.config.save_pretrained(save_path)

tokenizer.save_pretrained(save_path)

print("Model saved successfully")

Model saved successfully


# FinBERT Evaluation Summary

## Observations

- FinBERT achieved competitive performance on financial sentiment classification.
- Transformer-based contextual embeddings improved semantic understanding.
- Limited training data reduced generalization capability.
- FinBERT required significantly higher computational resources compared to RNN, LSTM, and GRU.

## Advantages

- Better contextual understanding
- Handles complex sentence structures
- Pretrained on financial text corpus
- Strong performance on domain-specific NLP tasks

## Limitations

- Longer training time
- Higher memory usage
- Requires larger datasets for optimal fine-tuning
- Sensitive to class imbalance

## Conclusion

FinBERT demonstrated strong potential for financial sentiment analysis. While GRU and LSTM achieved comparable accuracy on smaller datasets, transformer-based architectures are more scalable and effective for large-scale NLP applications.